In [ ]:
import pandas as pd
import geopandas as gpd

In [ ]:
students = pd.read_csv("./data/students.csv")
schools = pd.read_excel("./data/finalized_schools.xlsx")

schools = schools[schools["Status"].isna()]

In [ ]:
students_in_schools_with_status = students[students["School Code"].isin(schools["School Code"])]
students_in_schools_with_status.head()

In [ ]:
with open("./data/finalized_students.csv", "w") as f:
    students_in_schools_with_status.to_csv(f, index=False)

In [ ]:
df = students_in_schools_with_status.copy(deep=True)

del df['geometry']
df = df[(df['Geocodio Latitude'] != 0) & (df['Geocodio Longitude'] != 0)]

geometry = gpd.points_from_xy(df["Geocodio Longitude"], df["Geocodio Latitude"])
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4269")
gdf

In [ ]:

gdf.plot()

In [ ]:
school_districts = gpd.read_file("../data/census/school_district/tl_2025_25_unsd.shp")
framingham_district = school_districts[school_districts["NAME"].str.startswith("Framingham")]

framingham_district.plot()

In [ ]:
base = framingham_district.plot(color='white', edgecolor='black')
gdf.plot(ax=base, marker='o', color='red', markersize=5)

In [ ]:
within = [geo.within(framingham_district.geometry.iloc[0]) for geo in gdf.geometry]
within_students = gdf[within]

base = framingham_district.plot(color='white', edgecolor='black')
within_students.plot(ax=base, marker='o', color='red', markersize=5)

In [ ]:
percentage_within = len(within_students) / len(gdf)
print(f"{percentage_within:.2%} of students are within the Framingham school district.")

In [ ]:
# geocode all the schools

school_geometry = gpd.points_from_xy(schools["Geocodio Longitude"], schools["Geocodio Latitude"])
school_gdf = gpd.GeoDataFrame(schools, geometry=school_geometry, crs="EPSG:4269")

base = framingham_district.plot(color='white', edgecolor='black')
school_gdf.plot(ax=base, marker='o', color='black', markersize=5)

In [ ]:
# Calculate average distance to school for students within the district
# use the school code to find the corresponding school for each student, then calculate the distance

within_students = within_students.assign(
    school_geometry=within_students["School Code"].map(school_gdf.set_index("School Code").geometry)
)

In [ ]:
CRS = "EPSG:26986"  # Massachusetts State Plane Mainland
school_gdf = school_gdf.to_crs(CRS)

def get_distance_to_school(student_row):
    school_code = student_row["School Code"]
    school_row = school_gdf[school_gdf["School Code"] == school_code]
    if len(school_row) == 0:
        return None
    school_geometry = school_row.geometry.iloc[0]
    return student_row.geometry.distance(school_geometry)

in_meters_crs = within_students.to_crs("EPSG:26986")
in_meters_crs = in_meters_crs.assign(
    distance_to_school=in_meters_crs.apply(get_distance_to_school, axis=1)
)

In [ ]:
average_distance = in_meters_crs["distance_to_school"].mean()
print(f"Average distance to school for students within the district: {average_distance:.2f} meters")

In [ ]:
# get block groups within school district and their median income

income = pd.read_csv("./data/income.csv")
income.rename(columns={"Geo__geoid_": "GEOID", "SE_A14006_001": "income"}, inplace=True)
block_groups = gpd.read_file("../data/census/block_groups/tl_2025_25_bg.shp")

block_groups["GEOID"] = block_groups["GEOID"].astype(str)
income["GEOID"] = income["GEOID"].astype(str)

block_groups = block_groups.merge(income, left_on="GEOID", right_on="GEOID")
# combine based on GEOID

only_bg_inside = [geo.within(framingham_district.geometry.iloc[0]) for geo in block_groups.geometry]
within_block_groups = block_groups[only_bg_inside]

In [ ]:
# plot next to each other
import matplotlib.pyplot as plt

# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
fig_d, ax_d = plt.subplots()

In [ ]:
in_meters_crs['distance_miles'] = in_meters_crs['distance_to_school'] / 1609.344

in_meters_crs

In [ ]:
# Map of students based on distnace to school

framingham_district.to_crs(CRS).plot(ax=ax_d, color='white', edgecolor='black')
in_meters_crs.plot(ax=ax_d, column='distance_miles', cmap='Reds', marker='o', markersize=5, legend=True, legend_kwds={'label': "Distance to School (miles)"})
school_gdf.to_crs(CRS).plot(ax=ax_d, marker='x', color='black', markersize=50)

# ax_d.set_title("Students in Framingham by Distance to School")
ax_d.set_axis_off()

for x, y, label in zip(school_gdf.geometry.x, school_gdf.geometry.y, school_gdf["School Code"]):
    ax_d.annotate("", xy=(x, y), xytext=(3, 3), textcoords="offset points")

ax_d.plot()

In [ ]:
fig_d.savefig("students_distance_to_school.pdf", bbox_inches="tight",)

In [ ]:
within_block_groups.head()

In [ ]:
# plot median income by block group

fig_m, ax_m = plt.subplots()

within_block_groups["income_thousands"] = within_block_groups["income"] / 1000.0

framingham_district.to_crs(CRS).plot(ax=ax_m, color='white', edgecolor='black')
within_block_groups.to_crs(CRS).plot(ax=ax_m, column='income_thousands', cmap='Blues', legend=True, legend_kwds={'label': "Median Income ($ thousands)"})
# ax_m.set_title("Median Income by Block Group in Framingham")

ax_m.set_axis_off()
ax_m.plot()

In [ ]:
fig_m.savefig("median_income_by_block_group.pdf", bbox_inches="tight",)